# pyVideoTrans – Google Colab Edition

Traduci video da una lingua all'altra direttamente su Google Colab:
riconoscimento vocale (ASR) → traduzione sottotitoli → sintesi vocale (TTS) → video finale.

---

**Pipeline:**
```
Video input
  ↓  [Cella 4] ASR  – estrae audio → trascrive → SRT sorgente
  ↓  [Cella 5] Translate – SRT sorgente → SRT tradotto
  ↓  [Cella 6] TTS  – SRT tradotto → audio doppiato
  ↓  [Cella 7] Assembla – audio + video → MP4 finale
```
oppure esegui tutto in un colpo con la **Cella 8 (pipeline completa VTV)**.

---
> **Suggerimento GPU:** Menu **Runtime → Cambia tipo di runtime → GPU (T4)**  
> Velocizza drasticamente la trascrizione Whisper.

## Cella 1 – Installa ffmpeg e dipendenze di sistema

In [ ]:
import subprocess, sys

# ffmpeg è necessario per estrarre audio e assemblare il video finale
subprocess.run(["apt-get", "install", "-y", "-q", "ffmpeg", "libsndfile1"], check=True)

# Pacchetti Python (headless – nessun Qt/GUI)
PACKAGES = [
    # audio / video
    "pydub", "soundfile", "ffmpeg-python", "scipy", "ten-vad",
    # ASR
    "faster-whisper",
    "openai-whisper",   # needed for RECOGN_TYPE=1
    # TTS
    "edge-tts",
    # subtitles
    "srt",
    # translation / AI APIs
    "openai",
    "deepl",              # TRANSLATE_TYPE=16 "tenacity", "httpx", "aiohttp", "elevenlabs",
    "deepl",              # TRANSLATE_TYPE=16
    # utilities
    "requests", "tqdm", "psutil", "zhconv", "jieba", "regex",
    # notebook async compatibility
    "nest_asyncio",
]
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + PACKAGES)
print("✓ Dipendenze installate.")

## Cella 2 – Clona il repository e monta Google Drive

Il codice backend (`videotrans/`) viene scaricato da GitHub.  
Il tuo Google Drive viene montato in `/content/drive` per leggere i video e salvare i risultati.

In [ ]:
import os, subprocess, sys
from pathlib import Path

REPO_DIR = "/content/ipynbVideoTrans"

if not Path(REPO_DIR).exists():
    subprocess.run(
        ["git", "clone", "--depth", "1",
         "https://github.com/BaffoBello14/ipynbVideoTrans.git",
         REPO_DIR],
        check=True
    )
    print(f"✓ Repository clonato in {REPO_DIR}")
else:
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print(f"✓ Repository aggiornato.")

# Aggiungi il repo al path Python
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

# Monta Google Drive (opzionale – commenta se non ti serve)
try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    print("✓ Google Drive montato in /content/drive")
except Exception:
    print("ℹ  Google Drive non disponibile (ambiente non-Colab o già montato).")

## Cella 3 – Carica il video (scegli un metodo)

Esegui **una sola** delle tre celle qui sotto in base a dove si trova il tuo video.

In [ ]:
# ── Metodo A: Upload diretto dal tuo computer ──────────────────────────────
from google.colab import files
uploaded = files.upload()                         # si apre il selettore file
INPUT_VIDEO = "/content/" + list(uploaded.keys())[0]
print(f"✓ Video caricato: {INPUT_VIDEO}")

In [ ]:
# # ── Metodo B: Da Google Drive ──────────────────────────────────────────────
# # Modifica il percorso con la posizione del tuo file su Drive
# INPUT_VIDEO = "/content/drive/MyDrive/video/mio_video.mp4"
# print(f"Video da Drive: {INPUT_VIDEO}")

In [ ]:
# # ── Metodo C: Da URL (wget) ────────────────────────────────────────────────
# import subprocess
# VIDEO_URL   = "https://example.com/video.mp4"     # sostituisci con il tuo URL
# INPUT_VIDEO = "/content/video.mp4"
# subprocess.run(["wget", "-q", "-O", INPUT_VIDEO, VIDEO_URL], check=True)
# print(f"✓ Video scaricato: {INPUT_VIDEO}")

## Cella 4 – Configurazione

**Modifica qui** tutti i parametri prima di avviare il pipeline.

In [ ]:
import os
from pathlib import Path

# ── Percorsi ──────────────────────────────────────────────────────────────
# INPUT_VIDEO è già definito nella cella precedente.
# Per salvare su Drive, cambia OUTPUT_DIR:
OUTPUT_DIR = "/content/output"               # locale Colab
# OUTPUT_DIR = "/content/drive/MyDrive/video/output"   # su Google Drive

# ── Lingue ────────────────────────────────────────────────────────────────
SOURCE_LANG   = "it"     # lingua parlata nel video  (zh, en, it, ja, ko, fr, de, es, …)
TARGET_LANG   = "en"     # lingua di destinazione

# ── ASR (riconoscimento vocale) ───────────────────────────────────────────
#   0 = Faster-Whisper  (locale, raccomandato su Colab con GPU)
#   1 = OpenAI-Whisper  (locale)
#   5 = OpenAI Speech API  (remoto, richiede OPENAI_API_KEY)
#   6 = Gemini AI          (remoto, richiede GEMINI_API_KEY)
RECOGN_TYPE   = 0
WHISPER_MODEL = "large-v3-turbo"   # tiny | small | base | medium | large-v3-turbo | large-v3
USE_CUDA      = True               # True se hai abilitato la GPU T4 su Colab

# ── Traduzione ────────────────────────────────────────────────────────────
#   0  = Google Translate   (gratis, nessuna chiave)
#   1  = Microsoft Translate
#   3  = OpenAI ChatGPT     (richiede OPENAI_API_KEY)
#   4  = DeepSeek           (richiede DEEPSEEK_API_KEY)
#   5  = Gemini AI          (richiede GEMINI_API_KEY)
#  16  = DeepL              (richiede DEEPL_API_KEY)
TRANSLATE_TYPE = 0


# ── Qualità traduzione ───────────────────────────────────────────────────────
# Se usi un provider AI (DeepSeek, OpenAI, Gemini…) imposta True per inviare
# l'intero SRT con timestamp all'LLM: il contesto migliora drasticamente la qualità.
# Con Google/Microsoft deve restare False.
AI_SENDS_SRT   = TRANSLATE_TYPE in [3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 23, 24]

# Righe SRT per batch (solo per provider non-AI).
# Valori alti = più contesto, ma Google può troncare la risposta.
# Default originale: 5. Consigliato: 10-15.
TRANS_THREAD   = 10

# Righe SRT per batch quando si usa AI con AI_SENDS_SRT=True.
AITRANS_THREAD = 30

# Se True, invia l'intero testo del video come contesto globale all'AI
# (migliora coerenza terminologica ma usa più token).
AI_GLOBAL_CONTEXT = True

# ── TTS (sintesi vocale) ──────────────────────────────────────────────────
#   0  = Edge-TTS   (gratis, Microsoft, raccomandato)
#  15  = OpenAI TTS (richiede OPENAI_API_KEY)
#  29  = Google TTS (gratis)
TTS_TYPE      = 0
# Voce Edge-TTS: usa la cella Utilità per elencare le voci disponibili
# Esempi: "it-IT-ElsaNeural", "it-IT-DiegoNeural", "en-US-JennyNeural"
VOICE_ROLE    = "en-US-JennyNeural"
VOICE_RATE    = "+0%"    # velocità: "+20%" più veloce, "-20%" più lento
VOLUME        = "+0%"
PITCH         = "+0Hz"

# ── Sottotitoli nel video finale ──────────────────────────────────────────
#   0 = nessun sottotitolo, 1 = hard (bruciati), 2 = soft (selezionabili)
SUBTITLE_TYPE = 1

# ── Chiavi API opzionali ──────────────────────────────────────────────────
# Inseriscile qui oppure usa os.environ / Colab Secrets (chiave a forma di lucchetto)
OPENAI_API_KEY   = os.environ.get("OPENAI_API_KEY", "")
GEMINI_API_KEY   = os.environ.get("GEMINI_API_KEY", "")
DEEPSEEK_API_KEY = os.environ.get("DEEPSEEK_API_KEY", "")
DEEPL_API_KEY    = os.environ.get("DEEPL_API_KEY", "")

# ── Percorsi derivati (non modificare) ────────────────────────────────────
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)
_stem      = Path(INPUT_VIDEO).stem
SOURCE_SRT = str(Path(OUTPUT_DIR) / f"{SOURCE_LANG}.srt")   # matches trans_create.py naming
TARGET_SRT = str(Path(OUTPUT_DIR) / f"{TARGET_LANG}.srt")   # matches trans_create.py naming
DUBBED_WAV = str(Path(OUTPUT_DIR) / f"{_stem}_dubbed.wav")
OUTPUT_MP4 = str(Path(OUTPUT_DIR) / f"{_stem}_translated.mp4")

print(f"Input  : {INPUT_VIDEO}")
print(f"Output : {OUTPUT_DIR}")
print(f"Lingua : {SOURCE_LANG} → {TARGET_LANG}")
print(f"ASR    : type={RECOGN_TYPE}, model={WHISPER_MODEL}, cuda={USE_CUDA}")
print(f"Trans  : type={TRANSLATE_TYPE}")
print(f"TTS    : type={TTS_TYPE}, voce={VOICE_ROLE}")

## Cella 5 – Inizializza il backend pyVideoTrans

In [ ]:
import sys, os
from pathlib import Path

REPO_DIR = "/content/ipynbVideoTrans"
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

import subprocess, importlib
# Pull latest fixes from the repository before importing
subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=False)
importlib.invalidate_caches()
# Reload key modules so fixes take effect without kernel restart
for _mod_name in list(__import__('sys').modules.keys()):
    if _mod_name.startswith('videotrans'):
        __import__('sys').modules.pop(_mod_name)



from videotrans.configure import config
config.init_run()

from videotrans.configure.config import ROOT_DIR, TEMP_DIR, app_cfg, settings, params, logger

# Espone le chiavi API al layer dei parametri interni
if OPENAI_API_KEY:
    params['openaitts_key']       = OPENAI_API_KEY
    params['chatgpt_key']         = OPENAI_API_KEY
    params['openairecognapi_key'] = OPENAI_API_KEY
if GEMINI_API_KEY:
    params['gemini_key']          = GEMINI_API_KEY
if DEEPSEEK_API_KEY:
    params['deepseek_key']        = DEEPSEEK_API_KEY
if DEEPL_API_KEY:
    params['deepl_key']           = DEEPL_API_KEY

app_cfg.exit_soft = False
app_cfg.exec_mode = 'cli'
# Applica le impostazioni di qualità traduzione
settings['trans_thread']     = TRANS_THREAD
settings['aitrans_thread']   = AITRANS_THREAD
settings['aisendsrt']        = AI_SENDS_SRT
settings['aitrans_context']  = AI_GLOBAL_CONTEXT


from videotrans.util import tools
from videotrans.util.gpus import getset_gpu
getset_gpu()

# Verifica CUDA
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    print(f"CUDA disponibile: {cuda_ok}" + (f" – {torch.cuda.get_device_name(0)}" if cuda_ok else ""))
    if USE_CUDA and not cuda_ok:
        print("⚠  USE_CUDA=True ma nessuna GPU trovata. Imposta USE_CUDA=False oppure abilita la GPU dal menu Runtime.")
except ImportError:
    print("PyTorch non installato.")

print(f"ROOT_DIR : {ROOT_DIR}")
print("✓ Backend inizializzato.")

## Cella 6 – Pipeline completa (ASR → Traduzione → TTS → MP4)

Esegui questa cella per tradurre il video in un colpo solo.

In [ ]:
import uuid
from pathlib import Path
from videotrans.task.trans_create import TransCreate
from videotrans.task.taskcfg import TaskCfgVTT

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(INPUT_VIDEO).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

vtv_cfg = TaskCfgVTT(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    # Lingue
    source_language_code = SOURCE_LANG,
    target_language_code = TARGET_LANG,
    # ASR
    recogn_type          = RECOGN_TYPE,
    model_name           = WHISPER_MODEL,
    is_cuda              = USE_CUDA,
    remove_noise         = False,
    enable_diariz        = False,
    rephrase             = 0,
    fix_punc             = False,
    # Traduzione
    translate_type       = TRANSLATE_TYPE,
    # TTS
    tts_type             = TTS_TYPE,
    voice_role           = VOICE_ROLE,
    voice_rate           = VOICE_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = False,
    video_autorate       = False,
    align_sub_audio      = True,
    # Output
    subtitle_type        = SUBTITLE_TYPE,
    is_separate          = False,
    recogn2pass          = False,
    clear_cache          = True,
)

app_cfg.current_status = 'ing'
print("Avvio pipeline VTV completa…")

trk = TransCreate(cfg=vtv_cfg)
trk.prepare()
trk.recogn()
trk.trans()
trk.dubbing()
trk.align()
trk.assembling()
trk.task_done()

print(f"\n✓ Completato! I file sono in: {OUTPUT_DIR}")
import os
for f in os.listdir(OUTPUT_DIR):
    print(f"  {f}")

## Cella 7 – Scarica il risultato

Scarica il MP4 finale sul tuo computer (oppure salvalo su Drive se `OUTPUT_DIR` punta lì).

In [ ]:
import os
from pathlib import Path

# Cerca il file MP4 prodotto
mp4_files = list(Path(OUTPUT_DIR).glob("*.mp4"))

if mp4_files:
    try:
        from google.colab import files
        for mp4 in mp4_files:
            print(f"Download: {mp4}")
            files.download(str(mp4))
    except ImportError:
        print("File disponibili in:", OUTPUT_DIR)
        for f in mp4_files:
            print(f"  {f}")
else:
    print(f"Nessun MP4 trovato in {OUTPUT_DIR}. Controlla i log.")

---
## Celle avanzate (opzionali)

Le celle seguenti eseguono singoli step del pipeline in modo indipendente.

### Solo ASR – trascrivi il video in SRT

In [ ]:
import uuid
from pathlib import Path
from videotrans.task._speech2text import SpeechToText
from videotrans.task.taskcfg import TaskCfgSTT

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(INPUT_VIDEO).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

stt_cfg = TaskCfgSTT(
    **file_obj,
    uuid            = task_uuid,
    cache_folder    = cache_folder,
    target_dir      = OUTPUT_DIR,
    recogn_type     = RECOGN_TYPE,
    detect_language = SOURCE_LANG,
    model_name      = WHISPER_MODEL,
    is_cuda         = USE_CUDA,
    remove_noise    = False,
    enable_diariz   = False,
    rephrase        = 0,
    fix_punc        = False,
)

print(f"Trascrizione con {WHISPER_MODEL}…")
trk = SpeechToText(cfg=stt_cfg, out_format='srt')
trk.prepare()
trk.recogn()
trk.diariz()
trk.task_done()

import shutil
_candidates = list(Path(cache_folder).glob("*.srt"))
if _candidates:
    shutil.copy(str(_candidates[0]), SOURCE_SRT)
    print(f"✓ SRT sorgente salvato in: {SOURCE_SRT}")
    import srt
    subs = list(srt.parse(Path(SOURCE_SRT).read_text(encoding='utf-8')))
    print(f"  Totale sottotitoli: {len(subs)}")
    for s in subs[:5]:
        print(f"  [{s.index}] {s.content.strip()}")
else:
    print("Nessun SRT trovato. Controlla i log.")

### Solo Traduzione – traduci un SRT esistente

In [ ]:
import uuid
from pathlib import Path
from videotrans.task._translate_srt import TranslateSrt
from videotrans.task.taskcfg import TaskCfgSTS

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(SOURCE_SRT).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

sts_cfg = TaskCfgSTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    source_sub           = SOURCE_SRT,
    source_language_code = SOURCE_LANG,
    target_language_code = TARGET_LANG,
    translate_type       = TRANSLATE_TYPE,
)

print(f"Traduzione {SOURCE_LANG} → {TARGET_LANG}…")
trk = TranslateSrt(cfg=sts_cfg, out_format=0)
trk.trans()
trk.task_done()

import shutil
if sts_cfg.target_sub and Path(sts_cfg.target_sub).exists():
    shutil.copy(sts_cfg.target_sub, TARGET_SRT)
    print(f"✓ SRT tradotto salvato in: {TARGET_SRT}")
else:
    print(f"Controlla: {cache_folder}")

### Solo TTS – genera audio doppiato da un SRT

In [ ]:
import uuid, shutil
from pathlib import Path
from videotrans.task._dubbing import DubbingSrt
from videotrans.task.taskcfg import TaskCfgTTS

task_uuid    = uuid.uuid4().hex[:10]
cache_folder = str(Path(TEMP_DIR) / task_uuid)
Path(cache_folder).mkdir(parents=True, exist_ok=True)

file_obj = tools.format_video(Path(TARGET_SRT).absolute().as_posix())
file_obj.pop('uuid', None)       # avoid duplicate kwarg
file_obj.pop('target_dir', None)  # avoid duplicate kwarg

tts_cfg = TaskCfgTTS(
    **file_obj,
    uuid                 = task_uuid,
    cache_folder         = cache_folder,
    target_dir           = OUTPUT_DIR,
    target_sub           = TARGET_SRT,
    target_language_code = TARGET_LANG,
    tts_type             = TTS_TYPE,
    voice_role           = VOICE_ROLE,
    voice_rate           = VOICE_RATE,
    volume               = VOLUME,
    pitch                = PITCH,
    voice_autorate       = False,
    align_sub_audio      = True,
)

print(f"Sintesi vocale con {VOICE_ROLE}…")
trk = DubbingSrt(cfg=tts_cfg, out_ext='wav')
trk.dubbing()
trk.align()
trk.task_done()

if tts_cfg.target_wav_output and Path(tts_cfg.target_wav_output).exists():
    shutil.copy(tts_cfg.target_wav_output, DUBBED_WAV)
    print(f"✓ Audio doppiato salvato in: {DUBBED_WAV}")
else:
    print(f"Controlla: {cache_folder}")

---
## Utilità

In [ ]:
# Elenca le voci Edge-TTS disponibili filtrate per lingua
# Es: locale_filter="it-"  oppure  "en-US"
import asyncio, edge_tts

async def elenca_voci(locale_filter="it-"):
    voci = await edge_tts.list_voices()
    for v in voci:
        if locale_filter.lower() in v['ShortName'].lower():
            print(f"{v['ShortName']:45s}  gender={v['Gender']}")

await elenca_voci(locale_filter="it-")

In [ ]:
# Elenca tutti i provider ASR / Traduzione / TTS disponibili
from videotrans import recognition, translator, tts

print("=== Provider ASR ===")
for i, n in enumerate(recognition.RECOGN_NAME_LIST):
    print(f"  {i:>2}: {n}")

print("\n=== Provider Traduzione ===")
for i, n in enumerate(translator.TRANSLASTE_NAME_LIST):
    print(f"  {i:>2}: {n}")

print("\n=== Provider TTS ===")
for i, n in enumerate(tts.TTS_NAME_LIST):
    print(f"  {i:>2}: {n}")

In [ ]:
# Verifica GPU e ffmpeg
import subprocess, shutil, torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}", end="")
if torch.cuda.is_available():
    print(f" – {torch.cuda.get_device_name(0)}  ({torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB)")
else:
    print()

ff = shutil.which('ffmpeg')
print(f"ffmpeg  : {ff or 'NON TROVATO'}")
if ff:
    v = subprocess.run(["ffmpeg", "-version"], capture_output=True, text=True)
    print("  " + v.stdout.split('\n')[0])